# Baseline Models - AESO Price Spike Forecasting

This notebook builds simple baseline predictions for the Alberta `spike_lead_2` classification task.
The goal is to compare neural-network models against simple baselines before claiming improvement.


In [1]:
# 1. Imports and config
import json
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import norm
from sklearn.metrics import average_precision_score, f1_score, fbeta_score, precision_score, recall_score, roc_auc_score
from statsmodels.tsa.statespace.sarimax import SARIMAX

# These paths are relative to this notebook's folder: JorgeFolder/models/baseline/
SOURCE_DATA = Path("../../../Data/CSVs/aeso_merged_2020_2025.csv")
OUTPUT_DIR = Path("baseline_run")

if not SOURCE_DATA.exists():
    raise FileNotFoundError("Could not resolve ../../../Data/CSVs/aeso_merged_2020_2025.csv from the notebook folder.")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "spike_lead_2"
SPIKE_THRESHOLD = 200.0
TRAIN_END_EXCLUSIVE = pd.Timestamp("2023-11-06 00:00:00")
VAL_END_EXCLUSIVE = pd.Timestamp("2024-12-12 00:00:00")
TEST_START = VAL_END_EXCLUSIVE
HORIZON = 2
SEASONAL_PERIOD = 24
SEASONAL_SHIFT = SEASONAL_PERIOD - HORIZON
THRESHOLD_GRID = np.linspace(0.05, 0.99, 95)
SARIMAX_ORDER = (3, 1, 1)

print(f"Source data    : {SOURCE_DATA}")
print(f"Output dir     : {OUTPUT_DIR}")
print(f"Target         : {TARGET}")
print(f"Forecast horizon: t+{HORIZON}")


Source data    : ..\..\..\Data\CSVs\aeso_merged_2020_2025.csv
Output dir     : baseline_run
Target         : spike_lead_2
Forecast horizon: t+2


## Baselines used
We evaluate four simple baselines:

- **Naive**: predict the future spike with the current observed spike state
- **Seasonal naive**: predict the future spike with the spike state from the same target hour on the previous day
- **Mean**: predict the training-set spike rate as a constant probability
- **SARIMAX + Fourier**: forecast the future price with a first-differenced SARIMAX model, then convert the price forecast into a spike probability

Because the target is `spike_lead_2`, every baseline is aligned to a two-hour-ahead prediction problem.


In [2]:
# 2. Load and prepare the target series
df = pd.read_csv(SOURCE_DATA, parse_dates=["datetime"])
df = df.sort_values("datetime").reset_index(drop=True)

required_cols = ["datetime", "ACTUAL_POOL_PRICE", TARGET]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

# We keep only rows where the t+2 target hour exists.
df["target_price_t_plus_2"] = df["ACTUAL_POOL_PRICE"].shift(-HORIZON)
df = df.dropna(subset=[TARGET, "target_price_t_plus_2"]).reset_index(drop=True)

# Current spike is observed at time t and is available when predicting t+2.
df["current_spike"] = (df["ACTUAL_POOL_PRICE"] > SPIKE_THRESHOLD).astype(int)
df[TARGET] = df[TARGET].astype(int)

df["naive_prob"] = df["current_spike"].astype(float)
df["seasonal_naive_prob"] = df["current_spike"].shift(SEASONAL_SHIFT).astype(float)

print(f"Rows available : {len(df):,}")
print(f"Spike rate     : {df[TARGET].mean():.4f}")


Rows available : 48,885
Spike rate     : 0.0994


In [3]:
# 3. Train / validation / test split
train = df[df["datetime"] < TRAIN_END_EXCLUSIVE].reset_index(drop=True)
validation = df[(df["datetime"] >= TRAIN_END_EXCLUSIVE) & (df["datetime"] < VAL_END_EXCLUSIVE)].reset_index(drop=True)
test = df[df["datetime"] >= TEST_START].reset_index(drop=True)

train_spike_rate = float(train[TARGET].mean())
validation["mean_prob"] = train_spike_rate
test["mean_prob"] = train_spike_rate

print(f"Train rows      : {len(train):,}")
print(f"Validation rows : {len(validation):,}")
print(f"Test rows       : {len(test):,}")
print(f"Train spike rate: {train_spike_rate:.4f}")


Train rows      : 33,696
Validation rows : 9,648
Test rows       : 5,541
Train spike rate: 0.1237


## SARIMAX walk-forward forecasting
The SARIMAX model treats the price series as having a unit root, so the model uses first differencing (`d = 1`).
We use a SARIMAX(3,1,1) specification with Fourier terms for daily and weekly seasonality.

To avoid leakage, each forecast for hour `t+2` uses only information available up to hour `t`:
- the model is fitted on the training window only
- for validation and test predictions, the state is updated sequentially with observed prices up to the current hour
- the two-step-ahead forecast uses only future calendar-based Fourier terms, which are known in advance


In [4]:
# 4. SARIMAX + Fourier baseline for price, converted into spike probabilities
# This produces a leak-free walk-forward forecast for price at t+2.

dt = df["datetime"]
hour = dt.dt.hour + dt.dt.minute / 60.0
dow_hour = dt.dt.dayofweek * 24 + dt.dt.hour

fourier_df = pd.DataFrame(
    {
        "sin_day_1": np.sin(2 * np.pi * hour / 24),
        "cos_day_1": np.cos(2 * np.pi * hour / 24),
        "sin_day_2": np.sin(4 * np.pi * hour / 24),
        "cos_day_2": np.cos(4 * np.pi * hour / 24),
        "sin_week_1": np.sin(2 * np.pi * dow_hour / (24 * 7)),
        "cos_week_1": np.cos(2 * np.pi * dow_hour / (24 * 7)),
    }
)

train_mask = df["datetime"] < TRAIN_END_EXCLUSIVE
eval_mask = df["datetime"] >= TRAIN_END_EXCLUSIVE

y_full = df["ACTUAL_POOL_PRICE"].astype(float)

sarimax_model = SARIMAX(
    y_full.loc[train_mask],
    exog=fourier_df.loc[train_mask],
    order=SARIMAX_ORDER,
    trend="n",
    enforce_stationarity=False,
    enforce_invertibility=False,
)

sarimax_results = sarimax_model.fit(disp=False, maxiter=50)
last_included = int(np.flatnonzero(train_mask)[-1])
forecast_rows = []

eval_indices = np.flatnonzero(eval_mask)
for loop_position, idx in enumerate(eval_indices, start=1):
    if idx + HORIZON >= len(df):
        break

    # Before forecasting t+2, update the state only with observations available up to time t.
    if idx > last_included:
        new_endog = y_full.iloc[last_included + 1 : idx + 1]
        new_exog = fourier_df.iloc[last_included + 1 : idx + 1]
        sarimax_results = sarimax_results.extend(new_endog, exog=new_exog)
        last_included = idx

    future_exog = fourier_df.iloc[idx + 1 : idx + HORIZON + 1]
    forecast_object = sarimax_results.get_forecast(steps=HORIZON, exog=future_exog)

    forecast_price = float(forecast_object.predicted_mean.iloc[-1])
    forecast_se = float(forecast_object.se_mean.iloc[-1])
    safe_se = max(forecast_se, 1e-6)

    spike_prob = float(norm.sf((SPIKE_THRESHOLD - forecast_price) / safe_se))

    forecast_rows.append(
        {
            "datetime": df.loc[idx, "datetime"],
            "sarimax_price_forecast": forecast_price,
            "sarimax_price_se": forecast_se,
            "sarimax_prob": spike_prob,
        }
    )

    if loop_position % 2000 == 0:
        print(f"Processed {loop_position:,} walk-forward forecasts...")

sarimax_forecasts_df = pd.DataFrame(forecast_rows)
validation = validation.merge(sarimax_forecasts_df, on="datetime", how="left")
test = test.merge(sarimax_forecasts_df, on="datetime", how="left")

print(f"SARIMAX forecasts in validation: {validation['sarimax_prob'].notna().sum():,}")
print(f"SARIMAX forecasts in test      : {test['sarimax_prob'].notna().sum():,}")


Processed 2,000 walk-forward forecasts...


Processed 4,000 walk-forward forecasts...


Processed 6,000 walk-forward forecasts...


Processed 8,000 walk-forward forecasts...


Processed 10,000 walk-forward forecasts...


Processed 12,000 walk-forward forecasts...


Processed 14,000 walk-forward forecasts...


SARIMAX forecasts in validation: 9,648
SARIMAX forecasts in test      : 5,539


In [5]:
# 5. Validation threshold tuning for each baseline
# Every baseline is scored the same way: choose the threshold on validation by F1, then apply it to test.

baseline_specs = [
    {"baseline": "Naive", "prob_col": "naive_prob"},
    {"baseline": "Seasonal naive (daily)", "prob_col": "seasonal_naive_prob"},
    {"baseline": "Mean", "prob_col": "mean_prob"},
    {"baseline": "SARIMAX + Fourier", "prob_col": "sarimax_prob"},
]

threshold_rows = []
best_baseline_rows = []
validation_predictions = []
test_predictions = []
comparison_rows = []

for spec in baseline_specs:
    baseline_name = spec["baseline"]
    prob_col = spec["prob_col"]

    val_cols = ["datetime", "ACTUAL_POOL_PRICE", TARGET, prob_col]
    test_cols = ["datetime", "ACTUAL_POOL_PRICE", TARGET, prob_col]

    if prob_col == "sarimax_prob":
        val_cols += ["sarimax_price_forecast", "sarimax_price_se"]
        test_cols += ["sarimax_price_forecast", "sarimax_price_se"]

    val_frame = validation[val_cols].copy().dropna().reset_index(drop=True)
    val_frame = val_frame.rename(columns={prob_col: "y_prob"})

    test_frame = test[test_cols].copy().dropna().reset_index(drop=True)
    test_frame = test_frame.rename(columns={prob_col: "y_prob"})

    for threshold in THRESHOLD_GRID:
        val_preds = (val_frame["y_prob"] >= threshold).astype(int)
        threshold_rows.append(
            {
                "baseline": baseline_name,
                "threshold": round(float(threshold), 2),
                "f1": float(f1_score(val_frame[TARGET], val_preds, zero_division=0)),
                "f2": float(fbeta_score(val_frame[TARGET], val_preds, beta=2, zero_division=0)),
                "precision": float(precision_score(val_frame[TARGET], val_preds, zero_division=0)),
                "recall": float(recall_score(val_frame[TARGET], val_preds, zero_division=0)),
                "predicted_positive_rate": float(val_preds.mean()),
            }
        )

    baseline_threshold_df = pd.DataFrame([r for r in threshold_rows if r["baseline"] == baseline_name]).sort_values(
        ["f1", "f2", "precision", "threshold"], ascending=[False, False, False, False]
    ).reset_index(drop=True)

    best_row = baseline_threshold_df.iloc[0].to_dict()
    best_baseline_rows.append(best_row)
    selected_threshold = float(best_row["threshold"])

    val_frame["y_pred"] = (val_frame["y_prob"] >= selected_threshold).astype(int)
    test_frame["y_pred"] = (test_frame["y_prob"] >= selected_threshold).astype(int)
    val_frame["baseline"] = baseline_name
    test_frame["baseline"] = baseline_name

    validation_predictions.append(val_frame)
    test_predictions.append(test_frame)

    try:
        val_roc_auc = float(roc_auc_score(val_frame[TARGET], val_frame["y_prob"]))
    except ValueError:
        val_roc_auc = np.nan
    try:
        test_roc_auc = float(roc_auc_score(test_frame[TARGET], test_frame["y_prob"]))
    except ValueError:
        test_roc_auc = np.nan

    comparison_rows.append(
        {
            "baseline": baseline_name,
            "selected_threshold": selected_threshold,
            "validation_f1": float(f1_score(val_frame[TARGET], val_frame["y_pred"], zero_division=0)),
            "validation_f2": float(fbeta_score(val_frame[TARGET], val_frame["y_pred"], beta=2, zero_division=0)),
            "validation_precision": float(precision_score(val_frame[TARGET], val_frame["y_pred"], zero_division=0)),
            "validation_recall": float(recall_score(val_frame[TARGET], val_frame["y_pred"], zero_division=0)),
            "validation_pr_auc": float(average_precision_score(val_frame[TARGET], val_frame["y_prob"])),
            "validation_roc_auc": val_roc_auc,
            "test_f1": float(f1_score(test_frame[TARGET], test_frame["y_pred"], zero_division=0)),
            "test_f2": float(fbeta_score(test_frame[TARGET], test_frame["y_pred"], beta=2, zero_division=0)),
            "test_precision": float(precision_score(test_frame[TARGET], test_frame["y_pred"], zero_division=0)),
            "test_recall": float(recall_score(test_frame[TARGET], test_frame["y_pred"], zero_division=0)),
            "test_pr_auc": float(average_precision_score(test_frame[TARGET], test_frame["y_prob"])),
            "test_roc_auc": test_roc_auc,
        }
    )

threshold_search_df = pd.DataFrame(threshold_rows)
best_thresholds_df = pd.DataFrame(best_baseline_rows).sort_values("f1", ascending=False).reset_index(drop=True)
comparison_df = pd.DataFrame(comparison_rows).sort_values("test_f1", ascending=False).reset_index(drop=True)
validation_predictions_df = pd.concat(validation_predictions, ignore_index=True)
test_predictions_df = pd.concat(test_predictions, ignore_index=True)

print("Best validation thresholds by baseline:")
display(best_thresholds_df)


Best validation thresholds by baseline:


,baseline,threshold,f1,f2,precision,recall,predicted_positive_rate
0,SARIMAX + Fourier,0.41,0.576697,0.578735,0.573333,0.580101,0.062189
1,Naive,0.99,0.536256,0.536256,0.536256,0.536256,0.061464
2,Seasonal naive (daily),0.99,0.306914,0.306914,0.306914,0.306914,0.061464
3,Mean,0.12,0.115809,0.246672,0.061464,1.000000,1.000000


In [6]:
# 6. Final comparison table
# This table is the main result of the notebook.
comparison_df


,baseline,selected_threshold,validation_f1,validation_f2,validation_precision,validation_recall,validation_pr_auc,validation_roc_auc,test_f1,test_f2,test_precision,test_recall,test_pr_auc,test_roc_auc
0,Naive,0.99,0.536256,0.536256,0.536256,0.536256,0.316074,0.752943,0.383838,0.383838,0.383838,0.383838,0.158341,0.686315
1,SARIMAX + Fourier,0.41,0.576697,0.578735,0.573333,0.580101,0.574748,0.909524,0.380435,0.363825,0.411765,0.353535,0.308329,0.886291
2,Seasonal naive (daily),0.99,0.306914,0.306914,0.306914,0.306914,0.136796,0.630762,0.103093,0.101833,0.105263,0.101010,0.026695,0.542695
3,Mean,0.12,0.115809,0.246672,0.061464,1.000000,0.061464,0.500000,0.035106,0.083375,0.017867,1.000000,0.017867,0.500000


In [7]:
# 7. Test confusion counts for the strongest simple baselines
# This makes the classification errors easy to interpret in terms of TP, FP, TN, and FN.

confusion_rows = []
for baseline_name in ["Naive", "SARIMAX + Fourier"]:
    frame = test_predictions_df[test_predictions_df["baseline"] == baseline_name].copy()

    y_true = frame[TARGET].astype(int)
    y_pred = frame["y_pred"].astype(int)

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    confusion_rows.append(
        {
            "baseline": baseline_name,
            "TP": tp,
            "FP": fp,
            "TN": tn,
            "FN": fn,
            "test_f1": float(f1_score(y_true, y_pred, zero_division=0)),
            "test_precision": float(precision_score(y_true, y_pred, zero_division=0)),
            "test_recall": float(recall_score(y_true, y_pred, zero_division=0)),
        }
    )

confusion_counts_df = pd.DataFrame(confusion_rows)
confusion_counts_df


,baseline,TP,FP,TN,FN,test_f1,test_precision,test_recall
0,Naive,38,61,5381,61,0.383838,0.383838,0.383838
1,SARIMAX + Fourier,35,50,5390,64,0.380435,0.411765,0.353535


In [8]:
# 7. Save outputs
threshold_search_df.to_csv(OUTPUT_DIR / "validation_threshold_search.csv", index=False)
best_thresholds_df.to_csv(OUTPUT_DIR / "best_thresholds.csv", index=False)
comparison_df.to_csv(OUTPUT_DIR / "baseline_comparison.csv", index=False)
confusion_counts_df.to_csv(OUTPUT_DIR / "test_confusion_counts.csv", index=False)
validation_predictions_df.to_csv(OUTPUT_DIR / "validation_predictions.csv", index=False)
test_predictions_df.to_csv(OUTPUT_DIR / "test_predictions.csv", index=False)
sarimax_forecasts_df.to_csv(OUTPUT_DIR / "sarimax_walk_forward_forecasts.csv", index=False)

with open(OUTPUT_DIR / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(
        {
            "target": TARGET,
            "spike_threshold": SPIKE_THRESHOLD,
            "forecast_horizon": HORIZON,
            "seasonal_shift": SEASONAL_SHIFT,
            "sarimax_order": SARIMAX_ORDER,
            "train_spike_rate": round(float(train_spike_rate), 4),
            "best_validation_thresholds": best_thresholds_df.to_dict(orient="records"),
            "test_results": comparison_df.to_dict(orient="records"),
        },
        f,
        indent=2,
    )

print(f"Outputs saved to: {OUTPUT_DIR}")


Outputs saved to: baseline_run
